# Observability Signals for Test Instability\nDerive log/metric-style signals from run-level CI data to rank unstable tests.

In [ ]:
import pandas as pd\nfrom pathlib import Path

In [ ]:
path = Path('../data/processed/synthetic_ci_runs.csv')\nif not path.exists():\n    raise FileNotFoundError('Run feature_engineering/generate_synthetic_logs.py first.')\n\nruns = pd.read_csv(path)\nruns['timestamp'] = pd.to_datetime(runs['timestamp'], utc=True)\nruns['failed'] = (runs['pass_fail'] == 0).astype(int)\nruns.head()

In [ ]:
signals = (\n    runs.groupby('test_name', as_index=False)\n    .agg(\n        total_runs=('run_id', 'count'),\n        fail_rate=('failed', 'mean'),\n        retry_rate=('retry_count', 'mean'),\n        duration_mean_ms=('duration_ms', 'mean'),\n        duration_std_ms=('duration_ms', 'std'),\n        p95_duration_ms=('duration_ms', lambda s: float(s.quantile(0.95))),\n        network_latency_mean=('network_latency', 'mean'),\n        cpu_load_mean=('cpu_load', 'mean'),\n    )\n)\nsignals['duration_cv'] = signals['duration_std_ms'] / signals['duration_mean_ms']\nsignals['instability_score'] = (\n    0.40 * signals['fail_rate']\n    + 0.20 * signals['retry_rate']\n    + 0.20 * signals['duration_cv']\n    + 0.20 * (signals['network_latency_mean'] / signals['network_latency_mean'].max())\n)\nsignals = signals.sort_values('instability_score', ascending=False).reset_index(drop=True)\nsignals[['test_name', 'fail_rate', 'retry_rate', 'duration_cv', 'instability_score']].head(10)

In [ ]:
top3 = signals[['test_name', 'instability_score']].head(3)\nprint('Top unstable tests based on observability signals:')\nprint(top3.to_string(index=False))